1. Criação do SCHEMA da camada gold

In [0]:
%sql

USE CATALOG medalhao;
CREATE SCHEMA IF NOT EXISTS gold;

2. Definição do catálogo e SCHEMA

In [0]:
catalogo = "medalhao"
gold_db_name = "gold"


3. Importação das funções

In [0]:
from pyspark.sql import functions as F

 
Projeto 1: Visão Comercial e Volume de Produtos

Entrega 1 - Tabela Principal (gold.fat_vendas_comercial):

In [0]:
# Leitura das tabelas da camada Silver
df_itens_pedidos = spark.table(f"{catalogo}.silver.fat_itens_pedidos")
df_produtos = spark.table(f"{catalogo}.silver.dim_produtos")
df_pedido_total = spark.table(f"{catalogo}.silver.fat_pedido_total")

# Seleção das colunas necessárias para os cálculos
df_itens_base = df_itens_pedidos.select(
    "id_pedido",
    "id_item",
    "id_produto"
)

df_produtos_base = df_produtos.select(
    "id_produto",
    "categoria_produto"
)

df_pedido_total_base = df_pedido_total.select(
    "id_pedido",
    "data_pedido",
    "valor_total_pago_brl",
    "valor_total_pago_usd"
)
# Quantidade de itens por pedido
# Será usada para distribuir o valor total do pedido entre os itens
df_qtd_itens_pedido = (
    df_itens_pedidos
    .groupBy("id_pedido")
    .agg(
        F.count("*").alias("qtd_itens")
    )
)
# Associação dos itens com a categoria do produto
# Também trata categorias ausentes
df_itens_categoria = (
    df_itens_base
    .join(
        df_produtos_base,
        on="id_produto",
        how="left"
    )
    .withColumn(
        "categoria_produto",
        F.coalesce(F.col("categoria_produto"), F.lit("Categoria não informada"))
    )
)
# Base detalhada para cálculo das métricas comerciais
# Junta itens, pedidos e quantidade de itens por pedido
df_vendas_comercial_base = (
    df_itens_categoria
    .join(
        df_pedido_total_base,
        on="id_pedido",
        how="inner"
    )
    .join(
        df_qtd_itens_pedido,
        on="id_pedido",
        how="inner"
    )
    # Extrai ano e mês da data do pedido
    .withColumn("data_pedido", F.to_date("data_pedido"))
    .withColumn("ano_venda", F.year("data_pedido"))
    .withColumn("mes_venda", F.month("data_pedido"))
    # Rateia o valor total do pedido igualmente entre os itens
    .withColumn(
        "valor_item_brl",
        F.col("valor_total_pago_brl") / F.col("qtd_itens")
    )
    .withColumn(
        "valor_item_usd",
        F.col("valor_total_pago_usd") / F.col("qtd_itens")
    )
)

# Agregação final da fato comercial
# Consolida os indicadores por ano, mês e categoria
df_fat_vendas_comercial = (
    df_vendas_comercial_base
    .groupBy(
        "ano_venda",
        "mes_venda",
        "categoria_produto"
    )
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("*").alias("qtd_itens_vendidos"),
        F.round(F.sum("valor_item_brl"), 2).alias("receita_total_brl"),
        F.round(F.sum("valor_item_usd"), 2).alias("receita_total_usd")
    )
    # Calcula o ticket médio em BRL
    .withColumn(
        "ticket_medio_brl",
        F.round(
            F.col("receita_total_brl") / F.when(F.col("total_pedidos") != 0, F.col("total_pedidos")),
            2
        )
    )
    .select(
        "ano_venda",
        "mes_venda",
        "categoria_produto",
        "total_pedidos",
        "qtd_itens_vendidos",
        "receita_total_brl",
        "receita_total_usd",
        "ticket_medio_brl"
    )
    .orderBy(
        F.asc("ano_venda"),
        F.asc("mes_venda"),
        F.asc("categoria_produto")
    )
)

# Escrita na camada Gold
df_fat_vendas_comercial.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{gold_db_name}.fat_vendas_comercial")

Entrega 2 - Rankings Comerciais (Exibição via display()):

In [0]:
# Leitura das tabelas da camada Silver
df_itens_pedidos = spark.table(f"{catalogo}.silver.fat_itens_pedidos")
df_produtos = spark.table(f"{catalogo}.silver.dim_produtos")

# Junta os itens dos pedidos com a dimensão de produtos
# e calcula a quantidade vendida por ID do produto
# O agrupamento por id_produto evita repetição de um mesmo produto no ranking
df_vendas_produto = (
    df_itens_pedidos
    .join(
        df_produtos.select(
            "id_produto",
            "nome_produto",
            "categoria_produto"
        ),
        on="id_produto",
        how="left"
    )
    # Agrupa por produto para consolidar as vendas
    .groupBy("id_produto")
    .agg(
        # Mantém um nome válido para o produto, mesmo se houver valor nulo
        F.first(
            F.coalesce(F.col("nome_produto"), F.lit("Produto sem nome")),
            ignorenulls=True
        ).alias("nome_produto"),
        # Mantém uma categoria válida para o produto, mesmo se houver valor nulo
        F.first(
            F.coalesce(F.col("categoria_produto"), F.lit("Categoria não informada")),
            ignorenulls=True
        ).alias("categoria"),
        # Conta quantas vezes o produto apareceu nos itens dos pedidos
        F.count("*").alias("quantidade_vendida")
    )
)

# Top 5 produtos MAIS vendidos
# Ordena da maior para a menor quantidade vendida
# Em caso de empate, usa o id_produto como critério de desempate
df_top_5_mais_vendidos = (
    df_vendas_produto
    .orderBy(
        F.desc("quantidade_vendida"),
        F.asc("id_produto")
    )
    .limit(5)
)

# Top 5 produtos MENOS vendidos
# Ordena da menor para a maior quantidade vendida
# Em caso de empate, usa o id_produto como critério de desempate
df_top_5_menos_vendidos = (
    df_vendas_produto
    .orderBy(
        F.asc("quantidade_vendida"),
        F.asc("id_produto")
    )
    .limit(5)
)

# Exibição dos rankings em tela
print("Top 5 Produtos MAIS Vendidos")
display(df_top_5_mais_vendidos)

print("Top 5 Produtos MENOS Vendidos")
display(df_top_5_menos_vendidos)

Top 5 Produtos MAIS Vendidos


id_produto,nome_produto,categoria,quantidade_vendida
aca2eb7d00ea1a7b8ebd4e68314663af,Estante de Livros Luxo,moveis_decoracao,527
99a4788cb24856965c36a24e339b6058,Cobertor Cinza,cama_mesa_banho,488
422879e10f46682990de24d770e7f83d,Cortador de Grama Branco,ferramentas_jardim,484
389d119b48cf3043d311335e499d9c6b,Kit de Ferramentas Ultra,ferramentas_jardim,392
368c6c730842d78016ad823897a372db,Kit de Ferramentas Master,ferramentas_jardim,388


Top 5 Produtos MENOS Vendidos


id_produto,nome_produto,categoria,quantidade_vendida
00066f42aeeb9f3007548bb9d3f33c38,Loção Corporal Preto,perfumaria,1
00088930e925c41fd95ebfe695fd2655,Central Multimídia Avançado,automotivo,1
0009406fd7479715e4bef61dd91f2462,Toalha de Banho Premium,cama_mesa_banho,1
000d9be29b5207b54e86aa1b1ac54872,Relógio Analógico Dourado,relogios_presentes,1
0011c512eb256aa0dbbb544d8dffcf6e,Bateria Automotiva Luxo,automotivo,1


Projeto 2: Satisfação de Clientes e Qualidade de Parceiros

Entrega 1 - Tabela Principal (gold.fat_avaliacoes_clientes):

In [0]:
# Leitura das tabelas da camada Silver
df_avaliacoes_pedidos = spark.table(f"{catalogo}.silver.fat_avaliacoes_pedidos")
df_itens_pedidos = spark.table(f"{catalogo}.silver.fat_itens_pedidos")
df_produtos = spark.table(f"{catalogo}.silver.dim_produtos")
df_vendedores = spark.table(f"{catalogo}.silver.dim_vendedores")


# Seleção das colunas necessárias para os joins e agregações
df_avaliacoes_base = df_avaliacoes_pedidos.select(
    "id_avaliacao",
    "id_pedido",
    "nota_avaliacao"
)

df_itens_base = df_itens_pedidos.select(
    "id_pedido",
    "id_item",
    "id_produto",
    "id_vendedor"
)

df_produtos_base = df_produtos.select(
    "id_produto",
    "categoria_produto"
)

df_vendedores_base = df_vendedores.select(
    "id_vendedor",
    "nome_vendedor",
    "estado"
)
# Base principal de avaliações dos clientes
# Junta avaliação, item, produto e vendedor no mesmo DataFrame
df_avaliacoes_clientes_base = (
    df_avaliacoes_base
    .join(
        df_itens_base,
        on="id_pedido",
        how="inner"
    )
    .join(
        df_produtos_base,
        on="id_produto",
        how="left"
    )
    .join(
        df_vendedores_base,
        on="id_vendedor",
        how="left"
    )
)
# Tratamento de valores nulos nas dimensões de produto e vendedor
df_avaliacoes_clientes_tratada = (
    df_avaliacoes_clientes_base
    .withColumn(
        "categoria_produto",
        F.coalesce(F.col("categoria_produto"), F.lit("Categoria não informada"))
    )
    .withColumn(
        "nome_vendedor",
        F.coalesce(F.col("nome_vendedor"), F.lit("Vendedor não informado"))
    )
    .withColumn(
        "estado",
        F.coalesce(F.col("estado"), F.lit("Estado não informado"))
    )
)
# Remove duplicidades para evitar contagem repetida da mesma avaliação
# A deduplicação considera a combinação entre avaliação, categoria e vendedor
df_avaliacoes_clientes_dedup = (
    df_avaliacoes_clientes_tratada
    .select(
        "id_avaliacao",
        "nota_avaliacao",
        "categoria_produto",
        "nome_vendedor",
        "estado"
    )
    .dropDuplicates([
        "id_avaliacao",
        "categoria_produto",
        "nome_vendedor",
        "estado"
    ])
)
# Agregação final da fato de avaliações dos clientes
# Calcula volume, média, avaliações positivas, negativas e satisfação
df_fat_avaliacoes_clientes = (
    df_avaliacoes_clientes_dedup
    .groupBy(
        "categoria_produto",
        "nome_vendedor",
        "estado"
    )
    .agg(
        F.count("*").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.sum(
            F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)
        ).alias("total_avaliacoes_positivas"),
        F.sum(
            F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)
        ).alias("total_avaliacoes_negativas")
    )
    .withColumn(
        "percentual_satisfacao",
        F.round(
            (
                F.col("total_avaliacoes_positivas") /
                F.when(F.col("total_avaliacoes") != 0, F.col("total_avaliacoes"))
            ) * 100,
            2
        )
    )
    .select(
        "categoria_produto",
        "nome_vendedor",
        "estado",
        "total_avaliacoes",
        "avaliacao_media",
        "total_avaliacoes_positivas",
        "total_avaliacoes_negativas",
        "percentual_satisfacao"
    )
    .orderBy(
        F.asc("categoria_produto"),
        F.asc("nome_vendedor"),
        F.asc("estado")
    )
)

# Escrita da tabela final na camada Gold
df_fat_avaliacoes_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{gold_db_name}.fat_avaliacoes_clientes")

Entrega 2 - Rankings de Qualidade (Exibição via display()):

In [0]:
# Leitura das tabelas da camada Silver
df_avaliacoes = spark.table(f"{catalogo}.silver.fat_avaliacoes_pedidos")
df_itens_pedidos = spark.table(f"{catalogo}.silver.fat_itens_pedidos")
df_produtos = spark.table(f"{catalogo}.silver.dim_produtos")
df_vendedores = spark.table(f"{catalogo}.silver.dim_vendedores")


# Consolida a qualidade dos produtos com base nas avaliações dos pedidos
# Calcula a média das notas e a quantidade de avaliações por produto
df_qualidade_produtos = (
    df_avaliacoes
    .join(
        df_itens_pedidos.select("id_pedido", "id_produto"),
        on="id_pedido",
        how="inner"
    )
    .groupBy("id_produto")
    .agg(
        F.avg("nota_avaliacao").alias("media_nota"),
        F.count("id_avaliacao").alias("quantidade_avaliacoes")
    )
    .join(
        df_produtos.select("id_produto", "nome_produto", "categoria_produto"),
        on="id_produto",
        how="left"
    )
    .select(
        "id_produto",
        F.coalesce(F.col("nome_produto"), F.lit("Produto sem nome")).alias("nome_produto"),
        F.coalesce(F.col("categoria_produto"), F.lit("Categoria não informada")).alias("categoria"),
        F.round(F.col("media_nota"), 2).alias("media_nota"),
        "quantidade_avaliacoes"
    )
)

# Produto MAIS bem avaliado
# Critério: maior nota, maior quantidade de avaliações e menor id como desempate
df_produto_mais_bem_avaliado = (
    df_qualidade_produtos
    .orderBy(
        F.desc("media_nota"),
        F.desc("quantidade_avaliacoes"),
        F.asc("id_produto")
    )
    .limit(1)
)

# Produto MENOS bem avaliado
# Critério: menor nota, maior quantidade de avaliações e menor id como desempate
df_produto_menos_bem_avaliado = (
    df_qualidade_produtos
    .orderBy(
        F.asc("media_nota"),
        F.desc("quantidade_avaliacoes"),
        F.asc("id_produto")
    )
    .limit(1)
)


# Consolida a qualidade dos vendedores com base nas avaliações dos pedidos
# Calcula a média das notas e a quantidade de avaliações por vendedor
df_qualidade_vendedores = (
    df_avaliacoes
    .join(
        df_itens_pedidos.select("id_pedido", "id_vendedor"),
        on="id_pedido",
        how="inner"
    )
    .groupBy("id_vendedor")
    .agg(
        F.avg("nota_avaliacao").alias("media_nota"),
        F.count("id_avaliacao").alias("quantidade_avaliacoes")
    )
    .join(
        df_vendedores.select("id_vendedor", "nome_vendedor"),
        on="id_vendedor",
        how="left"
    )
    .select(
        "id_vendedor",
        F.coalesce(F.col("nome_vendedor"), F.lit("Vendedor sem nome")).alias("nome_vendedor"),
        F.round(F.col("media_nota"), 2).alias("media_nota"),
        "quantidade_avaliacoes"
    )
)

# Vendedor MAIS bem avaliado
# Critério: maior nota, maior quantidade de avaliações e menor id como desempate
df_vendedor_mais_bem_avaliado = (
    df_qualidade_vendedores
    .orderBy(
        F.desc("media_nota"),
        F.desc("quantidade_avaliacoes"),
        F.asc("id_vendedor")
    )
    .limit(1)
)

# Vendedor MENOS bem avaliado
# Critério: menor nota, maior quantidade de avaliações e menor id como desempate
df_vendedor_menos_bem_avaliado = (
    df_qualidade_vendedores
    .orderBy(
        F.asc("media_nota"),
        F.desc("quantidade_avaliacoes"),
        F.asc("id_vendedor")
    )
    .limit(1)
)


# Exibição dos resultados em tela
print("Produto MAIS bem avaliado")
display(df_produto_mais_bem_avaliado)

print("Produto MENOS bem avaliado")
display(df_produto_menos_bem_avaliado)

print("Vendedor MAIS bem avaliado")
display(df_vendedor_mais_bem_avaliado)

print("Vendedor MENOS bem avaliado")
display(df_vendedor_menos_bem_avaliado)

Produto MAIS bem avaliado


id_produto,nome_produto,categoria,media_nota,quantidade_avaliacoes
37eb69aca8718e843d897aa7b82f462d,Kit de Ferramentas Dourado,ferramentas_jardim,5.0,15


Produto MENOS bem avaliado


id_produto,nome_produto,categoria,media_nota,quantidade_avaliacoes
05b515fdc76e888aada3c6d66c201dff,Chapinha Plus,beleza_saude,1.0,10


Vendedor MAIS bem avaliado


id_vendedor,nome_vendedor,media_nota,quantidade_avaliacoes
48efc9d94a9834137efd9ea76b065a38,Luiz Otávio Abreu,5.0,34


Vendedor MENOS bem avaliado


id_vendedor,nome_vendedor,media_nota,quantidade_avaliacoes
8d92f3ea807b89465643c219455e7369,Sra. Fernanda Santos,1.0,8
